# SKU Health Analysis - Clean Portfolio Version

## Objective

This notebook analyzes SKU-level inventory health using sales, inventory, demand forecast, supplier lead time, cost, and price data.

The goal is to identify:

- Healthy SKUs
- Watch-list SKUs
- Potential risk SKUs
- High-impact SKUs that deserve priority attention

This version is designed for portfolio review: it should run from top to bottom and present a clear business story.


## 1. Setup and Load Data

The notebook uses a relative project path first. This makes the notebook easier to run after the project is downloaded from GitHub.


In [ ]:
from pathlib import Path
import pandas as pd


In [ ]:
candidate_paths = [Path("../01_Raw_Data/supply_chain_dataset1.csv"), Path("01_Raw_Data/supply_chain_dataset1.csv"), Path("/Users/gongwenkai/Projects/Portfolio/Inventory-Health-Analysis/01_Raw_Data/supply_chain_dataset1.csv")]
data_path = next((path for path in candidate_paths if path.exists()), None)


In [ ]:
if data_path is None:
    raise FileNotFoundError("Could not find supply_chain_dataset1.csv. Place it under 01_Raw_Data/ and run again.")


In [ ]:
df = pd.read_csv(data_path)
df.head()


## 2. Initial Data Check

Before analysis, confirm the dataset shape, column names, missing values, duplicates, and date range.


In [ ]:
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())


In [ ]:
quality_summary = pd.DataFrame({"missing_values": df.isna().sum(), "data_type": df.dtypes.astype(str)})
quality_summary


In [ ]:
print("Duplicate rows:", df.duplicated().sum())
df["Date"] = pd.to_datetime(df["Date"])


In [ ]:
print("Date range:", df["Date"].min().date(), "to", df["Date"].max().date())
df.describe().round(2)


In [ ]:
print("Promotion_Flag values:")
print(df["Promotion_Flag"].value_counts(dropna=False).sort_index())


In [ ]:
print("Stockout_Flag values:")
print(df["Stockout_Flag"].value_counts(dropna=False).sort_index())


## 3. Create Business Metrics

The raw dataset is transformed into business-friendly metrics that can support inventory and procurement decisions.


In [ ]:
df["Revenue"] = df["Units_Sold"] * df["Unit_Price"]
df["Inventory_Gap"] = df["Inventory_Level"] - df["Reorder_Point"]


In [ ]:
df["Forecast_Gap"] = df["Inventory_Level"] - df["Demand_Forecast"]
df["Gross_Margin_Per_Unit"] = df["Unit_Price"] - df["Unit_Cost"]


In [ ]:
df[["SKU_ID", "Date", "Units_Sold", "Inventory_Level", "Reorder_Point", "Demand_Forecast", "Revenue", "Inventory_Gap", "Forecast_Gap", "Gross_Margin_Per_Unit"]].head()


## 4. Build SKU-Level Summary

The raw records are aggregated from transaction/date level into one row per SKU.

Important note: `Promotion_Records` and `Stockout_Records` are record counts, not unique calendar days.


In [ ]:
sku_summary = df.groupby("SKU_ID").agg(
    Total_Units_Sold=("Units_Sold", "sum"),
    Total_Revenue=("Revenue", "sum"),
    Avg_Inventory=("Inventory_Level", "mean"),
    Avg_Reorder_Point=("Reorder_Point", "mean"),
    Avg_Inventory_Gap=("Inventory_Gap", "mean"),
    Avg_Forecast_Gap=("Forecast_Gap", "mean"),
    Avg_Lead_Time=("Supplier_Lead_Time_Days", "mean"),
    Avg_Unit_Cost=("Unit_Cost", "mean"),
    Avg_Unit_Price=("Unit_Price", "mean"),
    Avg_Gross_Margin_Per_Unit=("Gross_Margin_Per_Unit", "mean"),
    Promotion_Records=("Promotion_Flag", "sum"),
    Stockout_Records=("Stockout_Flag", "sum"),
).reset_index()


In [ ]:
sku_summary.head().round(2)


## 5. Classify SKU Health

The following rule-based framework is used for this portfolio project:

- `Risk`: average forecast gap is below 0
- `Watch`: average inventory gap is below 50
- `Watch`: average supplier lead time is above 10 days
- `Healthy`: none of the above conditions apply

In a real company, these thresholds should be adjusted based on service level targets, product category, and supplier reliability.


In [ ]:
def classify_sku(row):
    if row["Avg_Forecast_Gap"] < 0:
        return "Risk"
    if row["Avg_Inventory_Gap"] < 50:
        return "Watch"
    if row["Avg_Lead_Time"] > 10:
        return "Watch"
    return "Healthy"


In [ ]:
sku_summary["Health_Status"] = sku_summary.apply(classify_sku, axis=1)
sku_summary[["SKU_ID", "Total_Revenue", "Avg_Inventory_Gap", "Avg_Forecast_Gap", "Avg_Lead_Time", "Health_Status"]].head().round(2)


## 6. Key Findings

Summarize SKU health distribution and prioritize Watch/Risk SKUs by revenue impact.


In [ ]:
health_counts = sku_summary["Health_Status"].value_counts().reindex(["Healthy", "Watch", "Risk"], fill_value=0)
health_share = (health_counts / health_counts.sum()).rename("Share")


In [ ]:
health_overview = pd.concat([health_counts.rename("SKU_Count"), health_share], axis=1)
health_overview


In [ ]:
attention_skus = sku_summary[sku_summary["Health_Status"].isin(["Watch", "Risk"])].copy()
attention_skus = attention_skus.sort_values("Total_Revenue", ascending=False)


In [ ]:
attention_skus[["SKU_ID", "Health_Status", "Total_Revenue", "Avg_Lead_Time", "Avg_Inventory_Gap", "Avg_Forecast_Gap", "Stockout_Records"]].head(10).round(2)


In [ ]:
top_attention_sku = attention_skus.iloc[0]
print(f"Total SKUs analyzed: {len(sku_summary)}")
print(f"Healthy SKUs: {health_counts['Healthy']} ({health_share['Healthy']:.0%})")
print(f"Watch SKUs: {health_counts['Watch']} ({health_share['Watch']:.0%})")
print(f"Risk SKUs: {health_counts['Risk']} ({health_share['Risk']:.0%})")
print(f"Top attention SKU: {top_attention_sku['SKU_ID']} with revenue {top_attention_sku['Total_Revenue']:,.2f} and average lead time {top_attention_sku['Avg_Lead_Time']:.1f} days")


## 7. Business Interpretation

- Most SKUs are currently healthy under the rule-based framework.
- Watch SKUs are mainly driven by supplier lead time rather than negative forecast gap.
- High-revenue Watch SKUs should receive priority review because their business impact is larger if supply issues occur.
- In this dataset, `Stockout_Flag` is 0 for all records, so the analysis is a risk-screening exercise rather than proof of actual stockout events.

## Recommended Next Steps

1. Review supplier lead time stability for Watch SKUs.
2. Revisit reorder point and safety stock assumptions for high-revenue Watch SKUs.
3. Adjust health thresholds using company-specific service level targets.
